# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imranusmaneii/flyrank-ml-imran/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Scoring / ranking.** The decision this improves is "which page should an editor fix first?" The model's job is to produce a **position-adjusted CTR opportunity score** for every page — how far the page's observed CTR sits below what we would expect for its position tier and content type — and then **rank pages by how much they underperform that expectation**. The top of the ranking is the refresh queue: pages that are under-delivering for the slot they occupy.

This is not classification (this lane has no yes/no label yet) and not clustering (we are not discovering groups). It is a ranking/scoring task because the only thing that matters is the **order** of pages by opportunity.

In [1]:
import pandas as pd

# One row = one page. Drop pages with no real position data
# (avg_position == 0 means "no data", not rank zero — see the data dictionary).
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["avg_position"] > 0]

print("pages:", len(df), "| distinct pages:", df["content_id"].nunique())
print("ctr is a continuous rate (x100 percent); 0.76 = 0.76%")
df["ctr"].describe()

pages: 28795 | distinct pages: 28795
ctr is a continuous rate (x100 percent); 0.76 = 0.76%


count    28795.000000
mean         0.519662
std          3.232606
min          0.000000
25%          0.000000
50%          0.080000
75%          0.300000
max        100.000000
Name: ctr, dtype: float64

## 2. Target or proxy

**Proxy target:** `residual_score = actual_ctr − expected_ctr`, where `expected_ctr` is the average CTR of the page's `(position_tier, content_type)` group.

This is a **proxy, not editor-confirmed ground truth**. It answers "does this page under-deliver for its slot?", not "an editor confirms this page deserves a refresh". The long-term goal stays **Precision@50** — do the top-50 pages we rank actually deserve a refresh — but that requires real labels (editor-confirmed judgements) we do not have yet. Until then this proxy is the most honest signal available: it controls for position and content type before flagging a page.

In [2]:
# Check: the proxy target is a group mean, so tiny groups = noisy expectations.
grp = df.groupby(["position_tier", "content_type"])["ctr"]
print("groups:", grp.ngroups)
print("smallest group size:", grp.size().min())
grp.size().unstack(fill_value=0)

groups: 14
smallest group size: 3


content_type,comparison article,feedly article,keyword article
position_tier,,,
deep,0,15,1304
page_1,435,878,10501
page_3_5,79,99,7064
striking,180,181,6943
top_3,3,193,920


## 3. Success metric

**Interim metric: residual magnitude + rank stability.** The score itself is residual magnitude — the pages at the bottom of the ranking (most negative residual) are the biggest opportunities. Because true **Precision@50** needs editor labels we do not have yet, the interim check is **rank stability**: do the same pages keep ranking at the top across reasonable variations in how `expected_ctr` is computed? If the top of the list depends on the exact definition of "expected", the score is not yet ready to ship.

The check below compares two reasonable constructions of `expected_ctr` — per `position_tier` alone (the version a simple fixed rule would use) vs per `position_tier × content_type` (our proxy). Only **2 of the top-50 pages (4%)** survive the change, so the interim metric is telling us the ranking is definition-sensitive. That is exactly what it is for: pin down the definition (and later the labels) before the score goes operational.

In [3]:
# Rank stability: do the same top-50 pages appear under two constructions of expected_ctr?
def top_50(exp):
    residual = df["ctr"] - exp
    return set(df.loc[residual.sort_values(ascending=True).index[:50], "content_id"])

exp_tier_only = df.groupby("position_tier")["ctr"].transform("mean")
exp_joint     = df.groupby(["position_tier", "content_type"])["ctr"].transform("mean")

overlap = top_50(exp_tier_only) & top_50(exp_joint)
print("top-50 overlap (tier-only vs tier x content_type):", len(overlap), "of 50")
print("overlap %:", round(100 * len(overlap) / 50, 1))

top-50 overlap (tier-only vs tier x content_type): 2 of 50
overlap %: 4.0


## 4. The unit of analysis, as a real dataframe

**One row = one page.** The dataframe below is the lane: every page's observed CTR, the `expected_ctr` for its `(position_tier, content_type)` group, and the `residual_score` we rank by. It is sorted worst-first so the top rows are the biggest opportunities.

In [4]:
# Select the lane columns and compute the proxy target per (position_tier, content_type).
lane = df[["content_id", "client_id", "content_type", "position_tier",
           "avg_position", "impressions_90d", "ctr"]].copy()

lane["expected_ctr"] = lane.groupby(["position_tier", "content_type"])["ctr"].transform("mean")
lane["residual_score"] = lane["ctr"] - lane["expected_ctr"]
lane = lane.sort_values("residual_score")

print("rows:", len(lane), "| one row = one page")
lane.head()

rows: 28795 | one row = one page


,content_id,client_id,content_type,position_tier,avg_position,impressions_90d,ctr,expected_ctr,residual_score
11345,content_b5b92b3c155f,client_d4735e3a26,feedly article,top_3,3.0,2,0.0,9.909016,-9.909016
11418,content_5b584649c307,client_d4735e3a26,feedly article,top_3,2.0,3,0.0,9.909016,-9.909016
26289,content_805150f19e6d,client_d4735e3a26,feedly article,top_3,2.0,2,0.0,9.909016,-9.909016
2300,content_7d64c11f6fc0,client_d4735e3a26,feedly article,top_3,2.0,2,0.0,9.909016,-9.909016
25689,content_ec49cfde89da,client_d4735e3a26,feedly article,top_3,3.0,6,0.0,9.909016,-9.909016


## 5. Why ML beats a fixed rule here

Because `expected_ctr` varies **jointly** by position tier AND content type — a single threshold cannot see both at once.

The groupby table below uses the same numbers that build the proxy. Read it by column: a keyword article's expected CTR spans ~0.146% (deep) to ~1.275% (top_3) — roughly an **8.7x** swing from position alone. Read it by row: inside a single tier (`page_1`), expected CTR spans 0.132% (comparison article) to 3.350% (feedly article) — a **25.4x** swing from content type alone. A feedly article sitting at `page_3_5` (expected 3.836) is expected to out-pull a keyword article sitting at `top_3` (expected 1.275).

A fixed rule that thresholds on a tier average (say "top_3 pages are fine") flags none of the underperforming keyword pages and misclassifies low-CTR feedly pages that are actually fine for their type. The joint pattern is real but too tangled to write as an if-statement — that is what the model learns.

In [5]:
# Expected CTR varies jointly by position tier AND content type.
joint = lane.groupby(["position_tier", "content_type"])["ctr"].mean().unstack().round(3)
joint

# One number: within a single tier (page_1), the spread across content types.
page_1 = joint.loc["page_1"].dropna()
print("page_1 expected CTR spread (feedly / comparison):", round(page_1.max() / page_1.min(), 1), "x")

page_1 expected CTR spread (feedly / comparison): 25.4 x


## Self-check

Walk-through of the "what done looks like" checklist from the assignment card:

- [x] **Every section above is filled — markdown thinking AND the code that backs it.** Sections 1–5 each have a written answer plus a runnable cell that produces the numbers cited: lane type and the continuous score in §1, group sizes in §2, the rank-stability overlap in §3, the page-level dataframe in §4, and the joint groupby in §5.
- [x] **The notebook runs top to bottom with no errors.** Executed top-to-bottom via `jupyter nbconvert --execute`; every code cell shows real output.
- [x] **No client names, URLs, or private queries anywhere.** `client_id` is a pseudonym, no real URLs or queries appear, and the only URL is the skeleton's pre-existing public Colab badge.
- [x] **My claims use careful words: observed, measured, directional, decision-support.** The residual is described as a *proxy*; Precision@50 is stated as the *long-term goal that needs labels*; the ranking is framed as *decision-support* for editors, not a confirmed ground-truth verdict.
- [x] **Committed to my repo under `work/notebooks/`.** This file lives at `work/notebooks/w02_ml_task_framing.ipynb`.